In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import infercnvpy as cnv
import matplotlib.pyplot as plt
import gc


sc.settings.set_figure_params(dpi=300,figsize=(5, 5))
sc.logging.print_header()

In [ ]:
# data dir
input_dir = " "
output_dir = " "
gencode_ref_dir = " "
os.chdir(output_dir)

# cell color
cell_type_color = {"Epithelia":"#00CDD1","Fibroblast":"#377EB8","Endothelia":"#4DAF4A","Acinar_cell":"#984EA3","Pericyte&SMC":"#F29403","T&NK_cell":"#F781BF","B_cell":"#BC9DCC",
                   "Plasma_cell":"#A65628","Myeloid_cell":"#54B0E4","Mast_cell":"#222F75","Neutrophils":"#B2DF8A","Hepatocyte":"#1B9E77","Melanoma_cell":"#E3BE00","Neuron":"#FB9A99",
                   "Astrocyte":"#E7298A","Oligodendrocyte":"#910241","Osteoblastic_cell":"#48dc88","Alpha_cell":"#C0AFE0","Bela_cell":"#FDC44D"}



In [ ]:
# pre work
file_list = os.listdir(input_dir)
# file_list = ["BRCA"]
all_cell_type = ["T&NK_cell", "B_cell", "Plasma_cell", "Myeloid_cell", "Mast_cell", "Neutrophils"]

# data run
for file in file_list:
    print(f"################################## {file} process ##################################")

    ## data read
    if os.path.exists(f"{input_dir}/{file}/scRNA_tumor_annotation.h5"):
        scRNA_current = sc.read_h5ad(f"{input_dir}/{file}/scRNA_annotation_tumor_transform.h5ad")
        # scRNA_current = diopy.input.read_h5(file = f"{input_dir}/{file}/scRNA_tumor_annotation.h5")
        # scRNA_current = scRNA_current[scRNA_current.obs['cell_type'] != 'undefine', :]
        # sc.pp.normalize_total(scRNA_current, target_sum=1e4)
        # sc.pp.log1p(scRNA_current)

        gc.collect()

    elif file == "ANS":
        scRNA_current = sc.read_h5ad(f"{input_dir}/{file}/scRNA_annotation_transform.h5ad")
        # scRNA_current = scRNA_current[scRNA_current.obs['cell_type'] != 'undefine', :]
        scRNA_current = scRNA_current[scRNA_current.obs['tumor_status'] == 'Metastasis', :]
        gc.collect()

    else:
        continue 


    ## dir create
    output_file = f"{output_dir}/{file}"
    os.makedirs(output_file, exist_ok=True)
    os.chdir(output_file)

    ## ref cell type
    celltype_current = scRNA_current.obs.cell_type.unique().tolist()
    reference_cell_type = [cell_type for cell_type in celltype_current if cell_type in all_cell_type]

    ## chr ano
    cnv.io.genomic_position_from_gtf(adata=scRNA_current, gtf_file=gencode_ref_dir,gtf_gene_id='gene_name', inplace=True)

    ## infercnv analysis
    cnv.tl.infercnv(
        scRNA_current,
        reference_key="cell_type",
        reference_cat=reference_cell_type,
        window_size=100,
        n_jobs=24
        )
    
    ## cnv plot
    cnv.pl.chromosome_heatmap(scRNA_current, groupby="cell_type",save="scRNA_chromosome_heatmap.pdf")

    ## cnv score
    cnv.tl.pca(scRNA_current)
    cnv.pp.neighbors(scRNA_current)
    cnv.tl.leiden(scRNA_current)
    cnv.tl.umap(scRNA_current)
    cnv.tl.cnv_score(scRNA_current)

    cnv.pl.umap(scRNA_current,color="cnv_leiden",legend_loc="on data",show=False,)
    plt.savefig(f'{output_file}/scRNA_umap_cnv_cluster.png', dpi=3000)
    cnv.pl.umap(scRNA_current, color="cnv_score",show=False)
    plt.savefig(f'{output_file}/scRNA_umap_cnv_score.png', dpi=3000)
    cnv.pl.umap(scRNA_current, color="cell_type",legend_loc="on data",show=False)
    plt.savefig(f'{output_file}/scRNA_umap_cell_type.png', dpi=3000)
    sc.pl.umap(scRNA_current, color='cell_type',legend_loc="on data",show = False)
    plt.savefig(f'{output_file}/scRNA_umap_cell_type_origin.png', dpi=3000)
    sc.pl.umap(scRNA_current, color='cnv_score',show = False)
    plt.savefig(f'{output_file}/scRNA_umap_cnv_score_origin.png', dpi=3000)


    ## data save
    diopy.output.write_h5(scRNA_current, file = f"{output_file}/scRNA_infercnv.h5",save_X=False)
    scRNA_current.write_h5ad(f"{output_file}/scRNA_infercnv.h5ad", compression="gzip")

    ## space free
    del scRNA_current
    gc.collect()

